# Section 0: Imports & Setup


In [38]:
import pandas as pd
import numpy as np
import torch
import os
import random
from torch import nn
from torch.utils.data import Dataset, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    BertTokenizer, BertModel, BertForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
import nltk
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
from nltk.corpus import wordnet
from google.colab import drive
drive.mount('/content/drive')

try:
    import torch_xla as torch_xla_pkg
    import torch_xla.core.xla_model as xm
    if not hasattr(torch, "xla"):
        torch.xla = torch_xla_pkg
    _TORCH_XLA_AVAILABLE = True
except Exception:
    xm = None
    _TORCH_XLA_AVAILABLE = False


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [39]:
#seed
SEED = random.randint(0, 4294967295)
print(f"Random seed: {SEED}")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


Random seed: 2746317213


# Section 1: WELFake 15k Subset Creation


In [40]:
WELFAKE_PATH = "/content/drive/MyDrive/datasets/WELFake_processed.csv"
WELFAKE_SUBSET_PATH = "/content/drive/MyDrive/datasets/WELFake_15k_subset.csv"
WELFAKE_PERTURBED_PATH = "/content/drive/MyDrive/datasets/WELFake_15k_subset_perturbed.csv"

# Subset generation
# df_welfake = pd.read_csv(WELFAKE_PATH).dropna()

# df_subset, _ = train_test_split(
#     df_welfake,
#     train_size=15000,
#     random_state=SEED,
#     stratify=df_welfake['label']
# )

# print(f"Subset size: {len(df_subset)}")
# print(df_subset['label'].value_counts())

# df_subset.to_csv(WELFAKE_SUBSET_PATH, index=False)
# print(f"Saved to: {WELFAKE_SUBSET_PATH}")


# Section 2: Text Perturbation Module


In [41]:
import subprocess
import sys

# Install nlpaug automatically if it's missing (Colab compatible)
try:
    import nlpaug.augmenter.word as naw
except ImportError:
    print("Installing nlpaug...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nlpaug"])
    import nlpaug.augmenter.word as naw

class TextPerturber:
    def __init__(self, delete_prob=0.1, substitute_prob=0.15):
        self.delete_prob = delete_prob
        self.substitute_prob = substitute_prob
        
        # Switched back to WordNet backend because nlpaug's contextual embeddings 
        # is currently incompatible with the latest HuggingFace transformers tokenizers.
        self.aug_sub = naw.SynonymAug(aug_src='wordnet', aug_p=self.substitute_prob)
        self.aug_del = naw.RandomWordAug(action="delete", aug_p=self.delete_prob)

    def perturb(self, text: str) -> str:
        # 1. Substitute words using WordNet synonyms
        augmented_text = self.aug_sub.augment(text)
        if isinstance(augmented_text, list):
            augmented_text = augmented_text[0]
            
        # 2. Randomly delete some words
        augmented_text = self.aug_del.augment(augmented_text)
        if isinstance(augmented_text, list):
            augmented_text = augmented_text[0]
            
        return augmented_text


In [42]:
# Section 2.1: Precompute Perturbed Texts (Run once)
PRECOMPUTE_PERTURBATIONS = False

if PRECOMPUTE_PERTURBATIONS:
    if os.path.exists(WELFAKE_PERTURBED_PATH):
        print(f"Perturbed file already exists: {WELFAKE_PERTURBED_PATH}")
    else:
        df = pd.read_csv(WELFAKE_SUBSET_PATH).dropna()
        perturber = TextPerturber()
        df["combined_text_perturbed"] = [
            perturber.perturb(t) for t in df["combined_text"].tolist()
        ]
        df.to_csv(WELFAKE_PERTURBED_PATH, index=False)
        print(f"Saved perturbed dataset: {WELFAKE_PERTURBED_PATH}")


# Section 3: Dataset Classes


In [43]:
class FakeNewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx], dtype=torch.long) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
    def __len__(self):
        return len(self.labels)

class AdversarialFakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, perturber=None, perturbed_texts=None, max_length=128):
        if perturbed_texts is None:
            if perturber is None:
                raise ValueError("perturber is required when perturbed_texts is not provided")
            perturbed_texts = [perturber.perturb(t) for t in texts]

        self.orig_enc = tokenizer(texts, truncation=True,
                                  padding='max_length', max_length=max_length)
        self.pert_enc = tokenizer(perturbed_texts, truncation=True,
                                  padding='max_length', max_length=max_length)
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx], dtype=torch.long) for k, v in self.orig_enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        item['input_ids_pert'] = torch.tensor(self.pert_enc['input_ids'][idx], dtype=torch.long)
        item['attention_mask_pert'] = torch.tensor(self.pert_enc['attention_mask'][idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)


# Section 4: Model Architecture & Loss


In [44]:
class AdversarialBERT(nn.Module):
    def __init__(self, num_labels=2, dropout=0.1):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(768, num_labels)

    def gradient_checkpointing_enable(self, **kwargs):
        self.bert.gradient_checkpointing_enable(**kwargs)

    def gradient_checkpointing_disable(self):
        self.bert.gradient_checkpointing_disable()

    def forward(self, input_ids, attention_mask, token_type_ids=None, **kwargs):
        if token_type_ids is not None:
            out = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        else:
            out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_emb = out.last_hidden_state[:, 0, :]   # [CLS] token, shape [B, 768]
        logits = self.classifier(self.dropout(cls_emb))
        return logits, cls_emb

def adversarial_loss(logits_orig, logits_pert, labels, cls_orig, cls_pert, lambda_adv=0.5):
    ce = nn.CrossEntropyLoss()
    ce_loss = 0.5 * (ce(logits_orig, labels) + ce(logits_pert, labels))
    target = torch.ones(cls_orig.size(0), device=cls_orig.device)
    adv_loss = nn.CosineEmbeddingLoss()(cls_orig, cls_pert, target)
    return ce_loss + (lambda_adv * adv_loss)

# Section 5: Custom Trainer


In [45]:
from transformers import TrainerCallback

class LambdaSchedulerCallback(TrainerCallback):
    def __init__(self, max_lambda=0.5, warmup_ratio=0.1):
        self.max_lambda = max_lambda
        self.warmup_ratio = warmup_ratio
        self.trainer = None

    def on_step_begin(self, args, state, control, **kwargs):
        trainer = kwargs.get('trainer') or self.trainer
        if trainer is None:
            return

        if not hasattr(trainer, 'lambda_adv'):
            trainer.lambda_adv = 0.0

        total_steps = state.max_steps
        current_step = state.global_step

        if total_steps <= 0:
            return

        warmup_steps = total_steps * self.warmup_ratio
        if warmup_steps <= 0:
            trainer.lambda_adv = self.max_lambda
            return

        if current_step < warmup_steps:
            trainer.lambda_adv = self.max_lambda * (current_step / warmup_steps)
        else:
            trainer.lambda_adv = self.max_lambda

class AdversarialTrainer(Trainer):
    def __init__(self, *args, lambda_adv=0.5, **kwargs):
        super().__init__(*args, **kwargs)
        self.lambda_adv = lambda_adv

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels           = inputs.get('labels')
        input_ids_pert   = inputs.get('input_ids_pert')
        attn_mask_pert   = inputs.get('attention_mask_pert')

        # Concatenate original and perturbed inputs along the batch dimension
        combined_input_ids = torch.cat([inputs['input_ids'], input_ids_pert], dim=0)
        combined_attention_mask = torch.cat([inputs['attention_mask'], attn_mask_pert], dim=0)
        
        combined_token_type_ids = None
        if 'token_type_ids' in inputs:
            # Duplicate the token_type_ids for the perturbed half
            combined_token_type_ids = torch.cat([inputs['token_type_ids'], inputs['token_type_ids']], dim=0)

        # 1 Single Forward Pass
        combined_logits, combined_cls = model(
            input_ids=combined_input_ids, 
            attention_mask=combined_attention_mask, 
            token_type_ids=combined_token_type_ids
        )

        # Split outputs back into original and perturbed sets physically
        batch_size = labels.size(0)
        logits_orig, logits_pert = combined_logits[:batch_size], combined_logits[batch_size:]
        cls_orig, cls_pert = combined_cls[:batch_size], combined_cls[batch_size:]

        loss = adversarial_loss(logits_orig, logits_pert, labels, cls_orig, cls_pert, self.lambda_adv)
        return (loss, (loss, logits_orig)) if return_outputs else loss

In [46]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary',
    )
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

def load_device():
    if _TORCH_XLA_AVAILABLE and xm is not None:
        try:
            device = xm.xla_device()
            print(f"✓ Using TPU: {device}")
            return device, True
        except Exception:
            pass
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"✓ Using CUDA: {torch.cuda.get_device_name(0)}")
    else:
        device = torch.device("cpu")
        print("⚠ Using CPU (Training will be slow!)")
    return device, False


# Section 6: Smoke Test (Sanity Check)


In [47]:
# Quick test to ensure adversarial training passes gradients correctly
# Set to True to run
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    print("Running Smoke Test...")
    device, use_tpu = load_device()
    df = pd.read_csv(WELFAKE_SUBSET_PATH).dropna().head(200)
    
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    perturber = TextPerturber()
    
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        df['combined_text'].tolist(), df['label'].tolist(),
        test_size=0.20, random_state=SEED, stratify=df['label']
    )
    
    train_dataset = AdversarialFakeNewsDataset(train_texts, train_labels, tokenizer, perturber)
    val_dataset = AdversarialFakeNewsDataset(val_texts, val_labels, tokenizer, perturber)
    
    model = AdversarialBERT()
    if not use_tpu:
        model.to(device)
        
    training_kwargs = {
        "output_dir": "./results_smoke",
        "num_train_epochs": 1,
        "per_device_train_batch_size": 16,
        "per_device_eval_batch_size": 16,
        "save_strategy": "no",
        "report_to": "none",
        "optim": "adamw_torch",
        "logging_steps": 5,
        "remove_unused_columns": False,
        "label_names": ["labels"],
        "gradient_checkpointing": False if use_tpu else True,
    }
    if "evaluation_strategy" in TrainingArguments.__init__.__code__.co_varnames:
        training_kwargs["evaluation_strategy"] = "no"
    else:
        training_kwargs["eval_strategy"] = "no"

    training_args = TrainingArguments(**training_kwargs)
    
    trainer = AdversarialTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        lambda_adv=0.5
    )
    
    trainer.train()
    print("Smoke test passed!")


# Experiment Setup


In [48]:
def run_experiment(exp_name, dataset_class, is_adversarial=False):
    device, use_tpu = load_device()
    df = pd.read_csv(WELFAKE_SUBSET_PATH).dropna()
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

    indices = list(range(len(df)))
    train_idx, val_idx = train_test_split(
        indices, test_size=0.20, random_state=SEED, stratify=df['label']
    )
    train_texts = [df['combined_text'].iloc[i] for i in train_idx]
    val_texts = [df['combined_text'].iloc[i] for i in val_idx]
    train_labels = [df['label'].iloc[i] for i in train_idx]
    val_labels = [df['label'].iloc[i] for i in val_idx]
    print(f"Train: {len(train_texts)} | Val: {len(val_texts)}")
    
    if is_adversarial:
        pert_texts = None
        if os.path.exists(WELFAKE_PERTURBED_PATH):
            pert_df = pd.read_csv(WELFAKE_PERTURBED_PATH).dropna()
            if ("combined_text_perturbed" in pert_df.columns) and (len(pert_df) == len(df)):
                pert_texts = pert_df["combined_text_perturbed"].tolist()
        train_pert_texts = [pert_texts[i] for i in train_idx] if pert_texts else None
        val_pert_texts = [pert_texts[i] for i in val_idx] if pert_texts else None

        perturber = None if pert_texts else TextPerturber()
        train_dataset = dataset_class(
            train_texts, train_labels, tokenizer, perturber,
            perturbed_texts=train_pert_texts
        )
        val_dataset = dataset_class(
            val_texts, val_labels, tokenizer, perturber,
            perturbed_texts=val_pert_texts
        )
        model = AdversarialBERT()
        TrainerClass = AdversarialTrainer
    else:
        train_encodings = tokenizer(train_texts, truncation=True, padding='max_length', max_length=128)
        val_encodings = tokenizer(val_texts, truncation=True, padding='max_length', max_length=128)
        train_dataset = dataset_class(train_encodings, train_labels)
        val_dataset = dataset_class(val_encodings, val_labels)
        model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
        TrainerClass = Trainer

    if not use_tpu:
        model.to(device)

    training_kwargs = {
        "output_dir": f"./results_{exp_name}",
        "num_train_epochs": 3,
        "per_device_train_batch_size": 32 if is_adversarial else 64,
        "per_device_eval_batch_size": 32 if is_adversarial else 64,
        "gradient_accumulation_steps": 2 if is_adversarial else 2,
        "save_strategy": "steps",
        "save_steps": 128,
        "save_total_limit": 2,
        "bf16": use_tpu,
        "gradient_checkpointing": (is_adversarial and not use_tpu),
        "report_to": "none",
        "optim": "adamw_torch",
        "logging_steps": 128,
        "metric_for_best_model": "f1",
        "load_best_model_at_end": True,
        "weight_decay": 0.01,
        "remove_unused_columns": False,
        "label_names": ["labels"],
    }
    if "evaluation_strategy" in TrainingArguments.__init__.__code__.co_varnames:
        training_kwargs["evaluation_strategy"] = "steps"
        training_kwargs["eval_steps"] = 128
    else:
        training_kwargs["eval_strategy"] = "steps"
        training_kwargs["eval_steps"] = 128

    training_args = TrainingArguments(**training_kwargs)

    if is_adversarial:
        trainer = TrainerClass(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=compute_metrics,
            callbacks=[
                EarlyStoppingCallback(early_stopping_patience=3),
                LambdaSchedulerCallback(max_lambda=0.5, warmup_ratio=0.1)
            ],
            lambda_adv=0.0
        )
        for callback in trainer.callback_handler.callbacks:
            if isinstance(callback, LambdaSchedulerCallback):
                callback.trainer = trainer
    else:
        trainer = TrainerClass(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
        )

    print(f"\n--- Running {exp_name} ---")
    trainer.train()
    
    val_metrics = trainer.evaluate()
    print(f"Validation Metrics: {val_metrics}")
    
    return model, trainer, val_metrics

In [ ]:
# # Warmup: compile a small adversarial run on TPU (speeds up first real epoch)
# RUN_WARMUP = True
# WARMUP_SAMPLES = 256

# if RUN_WARMUP:
#     device, use_tpu = load_device()
#     df_full = pd.read_csv(WELFAKE_SUBSET_PATH).dropna()
#     if WARMUP_SAMPLES < len(df_full):
#         df_warm, _ = train_test_split(
#             df_full, train_size=WARMUP_SAMPLES, random_state=SEED, stratify=df_full['label']
#         )
#     else:
#         df_warm = df_full

#     tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
#     perturber = TextPerturber()
#     warm_texts = df_warm['combined_text'].tolist()
#     warm_labels = df_warm['label'].tolist()
#     warm_dataset = AdversarialFakeNewsDataset(warm_texts, warm_labels, tokenizer, perturber)

#     model = AdversarialBERT()
#     if not use_tpu:
#         model.to(device)

#     warm_kwargs = {
#         "output_dir": "./results_warmup",
#         "num_train_epochs": 1,
#         "per_device_train_batch_size": 4 if use_tpu else 8,
#         "save_strategy": "no",
#         "report_to": "none",
#         "optim": "adamw_torch",
#         "logging_steps": 5,
#         "remove_unused_columns": False,
#         "label_names": ["labels"],
#         "bf16": use_tpu,
#         "gradient_checkpointing": False,
#     }
#     if "evaluation_strategy" in TrainingArguments.__init__.__code__.co_varnames:
#         warm_kwargs["evaluation_strategy"] = "no"
#     else:
#         warm_kwargs["eval_strategy"] = "no"

#     warm_args = TrainingArguments(**warm_kwargs)
#     warm_trainer = AdversarialTrainer(
#         model=model,
#         args=warm_args,
#         train_dataset=warm_dataset,
#         lambda_adv=0.5
#     )
#     warm_trainer.train()
#     print("Warmup run complete.")


# Section 7: Experiment 0 - Baseline BERT


In [55]:
base_model, base_trainer, base_val_metrics = run_experiment('Baseline', FakeNewsDataset, is_adversarial=False)


/tmp/ipykernel_21682/2398473430.py:18: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()


✓ Using TPU: xla:0
Train: 12000 | Val: 3000


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



--- Running Baseline ---


...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Validation Metrics: {'eval_loss': 0.069091796875, 'eval_accuracy': 0.976, 'eval_f1': 0.9734904270986745, 'eval_precision': 0.9756457564575646, 'eval_recall': 0.9713445995591476, 'eval_runtime': 1.612, 'eval_samples_per_second': 1866.029, 'eval_steps_per_second': 29.157, 'epoch': 3.0}


# Section 8: Experiment 1 - Adversarial BERT


In [49]:
adv_model, adv_trainer, adv_val_metrics = run_experiment('Adversarial', AdversarialFakeNewsDataset, is_adversarial=True)


/tmp/ipykernel_21682/2398473430.py:18: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()


✓ Using TPU: xla:0
Train: 12000 | Val: 3000


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Running Adversarial ---


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
128,0.394531,0.073851,0.975000,0.972777,0.961263,0.984570
256,0.210938,0.054566,0.981333,0.979412,0.980132,0.978692
384,0.150391,0.051062,0.980667,0.978614,0.982235,0.975018
512,0.138672,0.050168,0.981000,0.978975,0.982963,0.975018


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Validation Metrics: {'eval_loss': 0.05456607788801193, 'eval_accuracy': 0.9813333333333333, 'eval_f1': 0.9794117647058823, 'eval_precision': 0.9801324503311258, 'eval_recall': 0.9786921381337252, 'eval_runtime': 3.2783, 'eval_samples_per_second': 917.548, 'eval_steps_per_second': 28.673, 'epoch': 3.0}


# Section 9: Results Comparison


In [56]:
print("Baseline Results:")
print(base_val_metrics)

print("Adversarial Results:")
print(adv_val_metrics)



Baseline Results:
{'eval_loss': 0.069091796875, 'eval_accuracy': 0.976, 'eval_f1': 0.9734904270986745, 'eval_precision': 0.9756457564575646, 'eval_recall': 0.9713445995591476, 'eval_runtime': 1.612, 'eval_samples_per_second': 1866.029, 'eval_steps_per_second': 29.157, 'epoch': 3.0}
Adversarial Results:
{'eval_loss': 0.05456607788801193, 'eval_accuracy': 0.9813333333333333, 'eval_f1': 0.9794117647058823, 'eval_precision': 0.9801324503311258, 'eval_recall': 0.9786921381337252, 'eval_runtime': 3.2783, 'eval_samples_per_second': 917.548, 'eval_steps_per_second': 28.673, 'epoch': 3.0}


# Section 10: Cross-Dataset Generalization


In [52]:
def cross_dataset_evaluation(model, tokenizer, is_adversarial, all_datasets_paths, compute_metrics_fn):
    from transformers.modeling_outputs import SequenceClassifierOutput
    device, use_tpu = load_device()
    
    print("\n" + "=" * 60)
    print("CROSS-DATASET GENERALIZATION")
    print("=" * 60)

    # Wrapper to make custom models compatible with standard Trainer evaluation
    class ModelWrapper(nn.Module):
        def __init__(self, inner_model):
            super().__init__()
            self.inner_model = inner_model
        def forward(self, input_ids, attention_mask, labels=None, **kwargs):
            # Handle different return types (AdversarialBERT returns tuple, BertForSequenceClassification returns ModelOutput)
            outputs = self.inner_model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs[0] if isinstance(outputs, (tuple, list)) else outputs.logits
            
            loss = None
            if labels is not None:
                loss = nn.CrossEntropyLoss()(logits, labels)
            
            return SequenceClassifierOutput(loss=loss, logits=logits)

    # Wrap the model
    eval_model = ModelWrapper(model)
    
    results = {}

    for name, path in all_datasets_paths.items():
        if name == "WELFake": # Skip training dataset
            continue

        print(f"\nTesting on unseen dataset: {name} (Full Dataset)...")
        df = pd.read_csv(path).dropna()

        test_texts = df['combined_text'].tolist()
        test_labels = df['label'].tolist()

        encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)
        dataset = FakeNewsDataset(encodings, test_labels)

        eval_trainer = Trainer(
            model=eval_model,
            compute_metrics=compute_metrics_fn,
            args=TrainingArguments(
                output_dir="./temp_eval",
                remove_unused_columns=False,
                label_names=["labels"],
                per_device_eval_batch_size=32,
                report_to="none"
            )
        )

        metrics = eval_trainer.evaluate(eval_dataset=dataset)
        results[name] = metrics
        
        acc = metrics.get('eval_accuracy', metrics.get('accuracy', 0))
        f1 = metrics.get('eval_f1', metrics.get('f1', 0))
        
        print(f"  -> {name} Accuracy: {acc:.4f}, F1: {f1:.4f}")

    return results

DATASETS = {
    "WELFake": "/content/drive/MyDrive/datasets/WELFake_processed.csv",
    "FakeNewsNet": "/content/drive/MyDrive/datasets/FakeNewsNet_processed.csv",
    "Fake_News_Detection": "/content/drive/MyDrive/datasets/Fake_News_Detection_processed.csv",
    "ISOT": "/content/drive/MyDrive/datasets/ISOT_processed.csv",
    "Fake_News_Classification": "/content/drive/MyDrive/datasets/Fake_News_Classification_processed.csv"
}

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')


In [53]:
adv_cross_results = cross_dataset_evaluation(adv_model, tokenizer, True, DATASETS, compute_metrics)


/tmp/ipykernel_21682/2398473430.py:18: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()


✓ Using TPU: xla:0

CROSS-DATASET GENERALIZATION

Testing on unseen dataset: FakeNewsNet (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> FakeNewsNet Accuracy: 0.6988, F1: 0.8158

Testing on unseen dataset: Fake_News_Detection (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Detection Accuracy: 0.0860, F1: 0.1478

Testing on unseen dataset: ISOT (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> ISOT Accuracy: 0.9930, F1: 0.9923

Testing on unseen dataset: Fake_News_Classification (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Classification Accuracy: 0.0246, F1: 0.0360


In [57]:
base_cross_results = cross_dataset_evaluation(
    model=base_model, 
    tokenizer=tokenizer, 
    is_adversarial=False, 
    all_datasets_paths=DATASETS, 
    compute_metrics_fn=compute_metrics
)

/tmp/ipykernel_21682/2398473430.py:18: DeprecationWarning: Use torch_xla.device instead
  device = xm.xla_device()


✓ Using TPU: xla:0

CROSS-DATASET GENERALIZATION

Testing on unseen dataset: FakeNewsNet (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> FakeNewsNet Accuracy: 0.6405, F1: 0.7690

Testing on unseen dataset: Fake_News_Detection (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Detection Accuracy: 0.0956, F1: 0.1613

Testing on unseen dataset: ISOT (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> ISOT Accuracy: 0.9910, F1: 0.9901

Testing on unseen dataset: Fake_News_Classification (Full Dataset)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  -> Fake_News_Classification Accuracy: 0.0266, F1: 0.0362
